<h1 align="center">Feature Engineering - Mobile Phones Price Prediction <h1>

In [48]:
import pandas as pd
import numpy as np
from scipy.stats import iqr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

In [91]:
df = pd.read_excel('Processed_Flipdata.xlsx')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.rename(columns={'Prize': 'Price'}, inplace=True)
print('Shape:', df.shape)
df.head()

Shape: (541, 11)


,Model,Colour,Memory,RAM,Battery_,Rear Camera,Front Camera,AI Lens,Mobile Height,Processor_,Price
0,Infinix SMART 7,Night Black,64,4,6000,13MP,5MP,1,16.76,Unisoc Spreadtrum SC9863A1,7299
1,Infinix SMART 7,Azure Blue,64,4,6000,13MP,5MP,1,16.76,Unisoc Spreadtrum SC9863A1,7299
2,MOTOROLA G32,Mineral Gray,128,8,5000,50MP,16MP,0,16.64,Qualcomm Snapdragon 680,11999
3,POCO C50,Royal Blue,32,2,5000,8MP,5MP,0,16.56,Mediatek Helio A22,5649
4,Infinix HOT 30i,Marigold,128,8,5000,50MP,5MP,1,16.76,G37,8999


## Step 1: Data Exploration & Understanding

In [ ]:
df.info()
print()
df.describe()

In [ ]:
# Missing values check
print('Missing values:')
print(df.isnull().sum())

In [92]:
# Remove duplicate records
before = df.shape[0]
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
after = df.shape[0]
print(f'Duplicates removed: {before - after}  |  Rows before: {before}  →  Rows after: {after}')

Duplicates removed: 10  |  Rows before: 541  →  Rows after: 531


In [ ]:
# ── Distribution, Skewness & Kurtosis ────────────────────────────────────────
numCols  = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'Price']
colorSeq = px.colors.qualitative.Plotly

def skew_label(s):
    if s > 1:      return 'Highly Positively Skewed'
    elif s < -1:   return 'Highly Negatively Skewed'
    else:          return 'Approximately Symmetric'

def kurt_label(k):
    if k > 1:    return 'Leptokurtic (heavy tails, outlier-prone)'
    elif k < -1: return 'Platykurtic (light tails, fewer outliers)'
    else:        return 'Mesokurtic (normal-like tails)'

# Summary table
print(f"{'Feature':<15} {'Skewness':>10} {'Interpretation':<35} {'Kurtosis':>10} {'Interpretation'}")
print('-' * 100)
for col in numCols:
    s = df[col].skew()
    k = df[col].kurtosis()
    print(f"{col:<15} {s:>10.3f}  {skew_label(s):<33} {k:>10.3f}  {kurt_label(k)}")

# Individual histogram per feature
for i, col in enumerate(numCols):
    skewness   = df[col].skew()
    kurtosis   = df[col].kurtosis()
    mean_val   = df[col].mean()
    median_val = df[col].median()

    fig = px.histogram(df, x=col, nbins=20, title=f'Distribution of {col}', text_auto=True,
        color_discrete_sequence=[colorSeq[i]],
        hover_data={col: True},
        labels={col: col, 'count': 'Frequency'}
    )
    fig.update_traces(marker_line_color='black', marker_line_width=1)

    fig.add_vline(x=mean_val, line_dash='dash', line_color='black',
                  annotation_text=f'Mean: {mean_val:.1f}',
                  annotation_position='top right', annotation_font_size=11)
    fig.add_vline(x=median_val, line_dash='dot', line_color='dimgray',
                  annotation_text=f'Median: {median_val:.1f}',
                  annotation_position='top left', annotation_font_size=11)

    fig.add_annotation(
        xref='paper', yref='paper', x=0.99, y=0.97,
        text=(f"<b>Skewness:</b> {skewness:.3f}  →  {skew_label(skewness)}<br>"
              f"<b>Kurtosis :</b> {kurtosis:.3f}  →  {kurt_label(kurtosis)}"),
        showarrow=False, align='left', xanchor='right', yanchor='top',
        bgcolor='lightyellow', bordercolor='gray', borderwidth=1,
        font=dict(size=11)
    )
    fig.show()

In [54]:
# Box plots to spot outliers
numCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'Price']
dfMelted = df[numCols].melt(var_name='Feature', value_name='Value')

fig = px.box(
    dfMelted, x='Feature', y='Value', color='Feature',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Box Plots — Numerical Features',
    hover_data={'Value': True, 'Feature': True},
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

### Insights — Step 1: Data Exploration

#### Distribution & Market Trends

**Price**
- **Skewness (Highly Positive):** The price distribution has a long right tail — the majority of devices are budget-to-mid-range (Rs. 5,000–20,000), while a small cluster of premium flagships (Rs. 50,000+) pulls the mean well above the median. This reflects the real Indian smartphone market where volume is driven by affordable segments.
- **Outliers:** Box plot reveals multiple extreme high-price outliers beyond the upper whisker. These are genuine premium devices (Apple, Samsung flagship) — not data errors — and must be retained to avoid biasing the model against the premium segment.
- **Market implication:** Log-transform Price before training linear models to reduce the disproportionate influence of flagship prices on the loss function.

**RAM**
- **Skewness (Positive / Moderate):** RAM clusters at popular market tiers (4 GB, 6 GB, 8 GB). The right tail reflects premium devices with 12–16 GB. The mass market is clearly mid-range.
- **Outliers:** High-end outliers (12 GB+) correspond to flagship gaming or power-user phones, which also command the highest prices — RAM outliers are strong pricing signals.
- **Market implication:** RAM tiers align directly with price segments (Budget ≤ 3 GB, Mid 4–6 GB, Premium 8+ GB). Binning into tiers can help models capture this non-linear step-up pricing.

**Memory (Internal Storage)**
- **Skewness (Positive):** Storage follows a similar tier-based pattern — 64 GB and 128 GB dominate, with 256 GB and 512 GB forming the right tail.
- **Outliers:** Ultra-high storage (512 GB+) outliers appear in the upper whisker zone and correspond exclusively to premium-priced devices.
- **Market implication:** Each storage tier jump (64→128→256 GB) commands a visible price premium. Storage is a key marketed specification that consumers anchor pricing expectations to.

**Battery**
- **Skewness (Near Symmetric / Slight Positive):** Battery capacity is tightly concentrated between 4,000–5,000 mAh — the current industry standard. The distribution is the most symmetric among all features, indicating that battery is now a commoditised spec across all price segments.
- **Outliers:** A few devices extend below 3,000 mAh (compact phones) and above 6,000 mAh (power-user devices), but these are rare.
- **Market implication:** Battery alone is a weak price predictor — offering a large battery does not justify a price premium in today's market where 5,000 mAh is baseline. Do not use battery capacity as a standalone pricing lever.

**Mobile Height**
- **Skewness (Near Symmetric):** Physical dimensions are tightly distributed around industry-standard form factors. Very little variance, reflecting manufacturing convergence on ~160 mm device heights.
- **Outliers:** Minimal — a few compact or oversized devices sit outside the whiskers but have negligible impact on the model.
- **Market implication:** Mobile height has low predictive value for price and may be safely deprioritised during feature selection.

## Step 2: Data Cleaning & Preprocessing

In [113]:
# Outlier detection via 1.5×IQR rule on Price (keep all rows — real premium phones)
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
lowerFence = Q1 - 1.5 * IQR
upperFence = Q3 + 1.5 * IQR

df['Price_outlier'] = ((df['Price'] < lowerFence) | (df['Price'] > upperFence)).astype(int)

outlierCount = df['Price_outlier'].sum()
normalCount  = len(df) - outlierCount
print(f'1.5×IQR bounds: [{lowerFence:,.0f}, {upperFence:,.0f}]')
print(f'Extreme outliers flagged: {outlierCount}  |  Normal rows: {normalCount}  |  All rows kept.')

# Plotly bar chart — outlier vs normal count
outlierSummary = pd.DataFrame({
    'Category': ['Normal', 'Extreme Outlier (IQR)'],
    'Count': [normalCount, outlierCount]
})
fig = px.bar(
    outlierSummary, x='Category', y='Count', color='Category',
    color_discrete_map={'Normal': 'mediumseagreen', 'Extreme Outlier (IQR)': 'tomato'},
    title='Price Outlier Detection (1.5×IQR Rule) — All Rows Kept',
    text='Count',
    hover_data={'Category': True, 'Count': True}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.update_layout(showlegend=False)
fig.show()

1.5×IQR bounds: [-1,450, 27,268]
Extreme outliers flagged: 0  |  Normal rows: 527  |  All rows kept.


In [115]:
#Determine Outliers from Numerical Columns using scipy iqr and handle them
numCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'Price']
outlier_indices = {}    
for col in numCols:
    col_IQR = iqr(df[col])
    lower_bound = df[col].quantile(0.25) - 1.5 * col_IQR
    upper_bound = df[col].quantile(0.75) + 1.5 * col_IQR
    outlier_indices[col] = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index.tolist()
    outliers = df.loc[outlier_indices[col]]             #
    if outliers.shape[0]<5 and outliers.shape[0] > 0 and col != 'Price':  # Only remove outliers if less than 5 and not Price
        print(f'Number of Outliers in {col}: {outliers.shape[0]} (Displaying first 10 outliers)')
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)] 
        print('Outliers Removed')
    else:
        #replace outliers with median
        median_value = df[col].median()
        df.loc[outlier_indices[col], col] = median_value
        print(f'Replacing outliers in {col} with median value: {median_value}')
        
if any(len(indices) > 0 for indices in outlier_indices.values()):
    print('Outliers detected in the dataset.')   

# Bar chart — rows removed/Replaced per column
RemRepPerCol = {col: len(indices) for col, indices in outlier_indices.items()}
remrepDf = pd.DataFrame({
    'Column': list(RemRepPerCol.keys()),
    'Rows Adjusted': list(RemRepPerCol.values())
})
fig = px.bar(
    remrepDf, x='Column', y='Rows Adjusted', color='Column',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Outlier Adjustment per Numerical Column (1.5×IQR Rule)',
    text='Rows Adjusted',
    hover_data={'Column': True, 'Rows Adjusted': True}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.update_layout(showlegend=False, xaxis_title='Column', yaxis_title='Rows Adjusted')
fig.show()

Replacing outliers in Memory with median value: 128.0
Replacing outliers in RAM with median value: 6.0
Replacing outliers in Battery_ with median value: 5000.0
Replacing outliers in Mobile Height with median value: 16.71
Replacing outliers in Price with median value: 13499.0


In [101]:
# ── Camera MP extraction (robust string parser) ──────────────────────────────
def extract_mp(val):
    try:
        return float(str(val).replace('MP', '').strip())
    except ValueError:
        return 0.0

df['RearCamera_MP']  = df['Rear Camera'].apply(extract_mp)
df['FrontCamera_MP'] = df['Front Camera'].apply(extract_mp)
print('Camera extraction sample:')
print(df[['Rear Camera', 'RearCamera_MP', 'Front Camera', 'FrontCamera_MP']].head())


# ── Brand extraction from Model (first word, uppercased) ─────────────────────
df['Brand'] = df['Model'].apply(lambda x: str(x).split()[0].upper())
print(f'\nBrand extraction — {df["Brand"].nunique()} unique brands:')
print(df['Brand'].value_counts().head(10))



# ── Processor brand family extraction ────────────────────────────────────────
def processor_brand(proc):
    proc = str(proc).upper()
    if 'QUALCOMM' in proc or 'SNAPDRAGON' in proc:
        return 'Qualcomm'
    elif 'MEDIATEK' in proc or 'DIMENSITY' in proc or 'HELIO' in proc:
        return 'MediaTek'
    elif 'UNISOC' in proc or 'SPREADTRUM' in proc:
        return 'Unisoc'
    elif 'APPLE' in proc or 'BIONIC' in proc or 'IOS' in proc:
        return 'Apple'
    elif 'EXYNOS' in proc:
        return 'Samsung_Exynos'
    elif 'KIRIN' in proc:
        return 'HiSilicon_Kirin'
    else:
        return 'Other'

df['Processor_Brand'] = df['Processor_'].apply(processor_brand)
print('\nProcessor Brand distribution:')
print(df['Processor_Brand'].value_counts())

Camera extraction sample:
  Rear Camera  RearCamera_MP Front Camera  FrontCamera_MP
0        13MP           13.0          5MP             5.0
1        13MP           13.0          5MP             5.0
2        50MP           50.0         16MP            16.0
3         8MP            8.0          5MP             5.0
4        50MP           50.0          5MP             5.0

Brand extraction — 20 unique brands:
Brand
REALME      97
REDMI       74
INFINIX     65
VIVO        60
POCO        58
SAMSUNG     53
MOTOROLA    45
OPPO        16
TECNO       15
MICROMAX    13
Name: count, dtype: int64

Processor Brand distribution:
Processor_Brand
MediaTek          269
Qualcomm          129
Unisoc             59
Other              38
Samsung_Exynos     28
Apple               4
Name: count, dtype: int64


### Updates — Step 2: Data Cleaning & Preprocessing

- **Duplicate removal** is performed before any analysis to prevent inflated counts, biased distributions, and data leakage between train/test splits.
- **`extract_mp()` function** replaces regex extraction: it strips the `"MP"` suffix and casts to float, returning `0.0` for unparseable values instead of `NaN` — safer for downstream arithmetic and avoids silent null propagation.
- **Brand extraction** (`Model.split()[0].upper()`) captures the phone manufacturer from the model name string (e.g., `"SAMSUNG Galaxy A54"` → `"SAMSUNG"`), creating a new high-signal categorical feature.
- **`processor_brand()` function** maps the raw free-text `Processor_` column into six clean brand categories (Qualcomm, MediaTek, Unisoc, Apple, Samsung_Exynos, HiSilicon_Kirin, Other), dramatically reducing cardinality from hundreds of variants to a manageable, meaningful set.
- **No missing values detected** in the core numeric features, simplifying preprocessing; `extract_mp` returns `0.0` for edge-case strings to keep column integrity.

## Step 3: Visual Representation

### 3.1 Univariate Analysis

In [ ]:
# Univariate — Distribution histograms for numerical features
numUnivarCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'Price', 'RearCamera_MP', 'FrontCamera_MP']
colorSeq = px.colors.qualitative.Plotly

for i, col in enumerate(numUnivarCols):
    mean_val   = df[col].mean()
    median_val = df[col].median()
    skewness   = df[col].skew()
    kurtosis   = df[col].kurtosis()

    fig = px.histogram(
        df, x=col, nbins=25,
        title=f'Distribution of {col}',
        color_discrete_sequence=[colorSeq[i % len(colorSeq)]],
        hover_data={col: True},
        labels={col: col, 'count': 'Frequency'}
    )
    fig.update_traces(marker_line_color='black', marker_line_width=1)
    fig.add_vline(x=mean_val, line_dash='dash', line_color='black',
                  annotation_text=f'Mean: {mean_val:.1f}',
                  annotation_position='top right', annotation_font_size=10)
    fig.add_vline(x=median_val, line_dash='dot', line_color='dimgray',
                  annotation_text=f'Median: {median_val:.1f}',
                  annotation_position='top left', annotation_font_size=10)
    fig.add_annotation(
        xref='paper', yref='paper', x=0.99, y=0.97,
        text=(f"<b>Skewness:</b> {skewness:.3f}<br><b>Kurtosis:</b> {kurtosis:.3f}"),
        showarrow=False, align='left', xanchor='right', yanchor='top',
        bgcolor='lightyellow', bordercolor='gray', borderwidth=1, font=dict(size=10)
    )
    fig.show()

In [ ]:
# Univariate — Box plots for numerical features (outlier detection)
numUnivarCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'Price', 'RearCamera_MP', 'FrontCamera_MP']
dfMelted = df[numUnivarCols].melt(var_name='Feature', value_name='Value')

fig = px.box(
    dfMelted, x='Feature', y='Value', color='Feature',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Box Plots — Numerical Features (Outlier View)',
    hover_data={'Value': True, 'Feature': True},
    points='outliers'
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Univariate — Bar plots for categorical features
catUnivarCols = ['Processor_Brand', 'AI Lens']

for col in catUnivarCols:
    counts = df[col].value_counts().reset_index()
    counts.columns = [col, 'Count']
    meanCount = counts['Count'].mean()

    fig = px.bar(
        counts, x=col, y='Count', color=col,
        color_discrete_sequence=px.colors.qualitative.Plotly,
        title=f'Frequency — {col}',
        text='Count',
        hover_data={col: True, 'Count': True}
    )
    fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
    fig.add_hline(y=meanCount, line_dash='dash', line_color='black',
                  annotation_text=f'Mean: {meanCount:.0f}',
                  annotation_position='top right', annotation_font_size=10)
    fig.update_layout(showlegend=False, xaxis_tickangle=-20)
    fig.show()

# ── Top 5 Brands — group remaining as 'Others' ───────────────────────────────
top5 = df['Brand'].value_counts().head(5).index.tolist()
df['Brand_Grouped'] = df['Brand'].apply(lambda x: x if x in top5 else 'Others')

brandCounts = df['Brand_Grouped'].value_counts().reset_index()
brandCounts.columns = ['Brand', 'Count']

# Assign distinct color: 'Others' gets grey, top 5 get qualitative colors
colorMap = {brand: px.colors.qualitative.Plotly[i] for i, brand in enumerate(top5)}
colorMap['Others'] = 'lightgrey'

fig = px.bar(
    brandCounts, x='Brand', y='Count', color='Brand',
    color_discrete_map=colorMap,
    title='Top 5 Brands by Device Count (Others Grouped)',
    text='Count',
    hover_data={'Brand': True, 'Count': True}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.update_layout(showlegend=False, xaxis_tickangle=-20)
fig.show()

print(f'Top 5 Brands: {top5}')
print(df['Brand_Grouped'].value_counts())

#### Insights on Univariate Analysis
**Numerical Features:**
- **Price:** Highly right-skewed — majority of devices are budget to mid-range (Rs. 5,000–20,000). Box plot shows multiple high-value outliers (premium flagships). Market is volume-driven at lower price points.
- **RAM:** Discrete tier distribution (4/6/8 GB dominant). Right tail outliers (12–16 GB) mark gaming/flagship segment. Each tier jump represents a clear market segment boundary.
- **Memory:** 64 GB and 128 GB dominate. Box plot upper outliers (512 GB+) are exclusively premium devices — storage tier is a strong pricing signal.
- **RearCamera_MP:** Right-skewed — most phones ship 12–50 MP; 64/108 MP outliers correspond to premium photography-focused flagships.
- **FrontCamera_MP:** Less skewed than rear — selfie camera specs are more homogeneous across price bands (16–32 MP is standard across mid and premium).
- **Battery:** Near-symmetric (4,000–5,000 mAh norm). Minimal outliers — battery is a commoditised spec; not a standalone price differentiator.
- **Mobile Height:** Near-normal, tight distribution. Minimal outliers — form factor is constrained by manufacturing standards.

**Categorical Features:**
- **Processor Brand:** Qualcomm dominates volume; Apple has fewest devices but highest average price. MediaTek serves the mid-to-budget segment.
- **AI Lens:** Majority of devices have AI Lens enabled — it has become a standard mid-range feature, not a premium exclusive.
- **Brand:** Samsung and Xiaomi lead in device count, reflecting their broad portfolio coverage across all price segments.

### 3.2 Bivariate Analysis

In [ ]:
# Bivariate — Scatter plots: numerical features vs Price
numBivarCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'RearCamera_MP', 'FrontCamera_MP']

for i, col in enumerate(numBivarCols):
    mean_x = df[col].mean()
    mean_y = df['Price'].mean()

    fig = px.scatter(
        df, x=col, y='Price',
        color_discrete_sequence=[px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]],
        title=f'{col} vs Price',
        opacity=0.5,
        hover_data={col: True, 'Price': True},
        labels={col: col, 'Price': 'Price (Rs.)'}
    )
    fig.add_vline(x=mean_x, line_dash='dash', line_color='black',
                  annotation_text=f'Mean {col}: {mean_x:.1f}',
                  annotation_position='top right', annotation_font_size=10)
    fig.add_hline(y=mean_y, line_dash='dot', line_color='dimgray',
                  annotation_text=f'Mean Price: {mean_y:.1f}',
                  annotation_position='bottom right', annotation_font_size=10)
    fig.show()

In [ ]:
# Bivariate — Avg Price by categorical features
catBivarCols = ['Processor_Brand', 'AI Lens']

for col in catBivarCols:
    grp = df.groupby(col)['Price'].mean().reset_index().sort_values('Price', ascending=False)
    grp.columns = [col, 'AvgPrice']
    grandMean = grp['AvgPrice'].mean()

    fig = px.bar(
        grp, x=col, y='AvgPrice', color=col,
        color_discrete_sequence=px.colors.qualitative.Plotly,
        title=f'Avg Price by {col}',
        text=grp['AvgPrice'].apply(lambda v: f'Rs.{v:,.0f}'),
        hover_data={col: True, 'AvgPrice': ':.0f'},
        labels={col: col, 'AvgPrice': 'Avg Price (Rs.)'}
    )
    fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
    fig.add_hline(y=grandMean, line_dash='dash', line_color='black',
                  annotation_text=f'Mean: Rs.{grandMean:,.0f}',
                  annotation_position='top right', annotation_font_size=10)
    fig.update_layout(showlegend=False, xaxis_tickangle=-20)
    fig.show()

# Avg Price by Brand (Top 5 + Others)
brandPrice = df.groupby('Brand_Grouped')['Price'].mean().reset_index().sort_values('Price', ascending=False)
brandPrice.columns = ['Brand', 'AvgPrice']
grandMeanBrand = brandPrice['AvgPrice'].mean()

colorMapBrand = {brand: px.colors.qualitative.Plotly[i] for i, brand in enumerate(top5)}
colorMapBrand['Others'] = 'lightgrey'

fig = px.bar(
    brandPrice, x='Brand', y='AvgPrice', color='Brand',
    color_discrete_map=colorMapBrand,
    title='Avg Price by Brand (Top 5 + Others)',
    text=brandPrice['AvgPrice'].apply(lambda v: f'Rs.{v:,.0f}'),
    hover_data={'Brand': True, 'AvgPrice': ':.0f'},
    labels={'Brand': 'Brand', 'AvgPrice': 'Avg Price (Rs.)'}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.add_hline(y=grandMeanBrand, line_dash='dash', line_color='black',
              annotation_text=f'Mean: Rs.{grandMeanBrand:,.0f}',
              annotation_position='top right', annotation_font_size=10)
fig.update_layout(showlegend=False, xaxis_tickangle=-20)
fig.show()

#### Insights on Bivariate Analysis
- **RAM vs Price:** Strong positive step-up relationship — each RAM tier commands a significantly higher average price. Non-linear: jump from 8→12 GB is disproportionately large.
- **Memory vs Price:** Clear monotonic increase — 128 GB commands ~2× the average price of 64 GB devices.
- **RearCamera_MP vs Price:** Positive correlation — higher MP cameras are bundled with pricier devices, though MP alone doesn't fully explain price (sensor quality and software matter too).
- **Battery vs Price:** Weak relationship — large battery phones exist across all price bands; battery size alone does not drive premium pricing.
- **Processor Brand vs Price:** Apple >> Qualcomm > MediaTek > Unisoc in average price. Processor brand is one of the strongest categorical predictors of price tier.
- **AI Lens vs Price:** Devices with AI Lens have a noticeably higher average price — confirms it is bundled with mid-to-premium devices.

### 3.3 Multivariate Analysis

In [ ]:
# Correlation heatmap — all numeric features after cleaning
corrCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'RearCamera_MP', 'FrontCamera_MP', 'Price']
corrMatrix = df[corrCols].corr()

fig = px.imshow(
    corrMatrix, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Heatmap — Numeric Features vs Price'
)
fig.update_traces(hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.3f}<extra></extra>')
fig.show()

In [116]:
# Avg Price by RAM and Memory (after outlier handling)
ramPrice = df.groupby('RAM')['Price'].mean().reset_index().sort_values('RAM')
ramPrice.columns = ['Value', 'AvgPrice']

memPrice = df.groupby('Memory')['Price'].mean().reset_index().sort_values('Memory')
memPrice.columns = ['Value', 'AvgPrice']

for data, xlabel, title, color in [
    (ramPrice,  'RAM (GB)',    'Avg Price by RAM (GB)',    'mediumpurple'),
    (memPrice, 'Memory (GB)', 'Avg Price by Memory (GB)', 'cornflowerblue'),
]:
    meanVal = data['AvgPrice'].mean()
    fig = px.bar(
        data, x='Value', y='AvgPrice',
        color_discrete_sequence=[color],
        title=title,
        text=data['AvgPrice'].apply(lambda v: f'Rs.{v:,.0f}'),
        hover_data={'Value': True, 'AvgPrice': ':.0f'},
        labels={'Value': xlabel, 'AvgPrice': 'Avg Price (Rs.)'}
    )
    fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
    fig.add_hline(y=meanVal, line_dash='dash', line_color='black',
                  annotation_text=f'Mean: Rs.{meanVal:,.0f}',
                  annotation_position='top right', annotation_font_size=10)
    fig.show()

In [ ]:
# Avg Price by Processor Brand
procPrice = df.groupby('Processor_Brand')['Price'].mean().reset_index().sort_values('Price', ascending=False)
procPrice.columns = ['Processor_Brand', 'AvgPrice']
overallMean = procPrice['AvgPrice'].mean()

fig = px.bar(
    procPrice, x='Processor_Brand', y='AvgPrice',
    color='Processor_Brand',
    color_discrete_sequence=px.colors.qualitative.Bold,
    title='Avg Price by Processor Brand',
    text=procPrice['AvgPrice'].apply(lambda v: f'Rs.{v:,.0f}'),
    hover_data={'Processor_Brand': True, 'AvgPrice': ':.0f'},
    labels={'Processor_Brand': 'Processor Brand', 'AvgPrice': 'Avg Price (Rs.)'}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.add_hline(y=overallMean, line_dash='dash', line_color='black',
              annotation_text=f'Overall Mean: Rs.{overallMean:,.0f}',
              annotation_position='top right', annotation_font_size=10)
fig.update_layout(showlegend=False, xaxis_tickangle=-20)
fig.show()

#### Insights on Multivariate Analysis (Correlation Map)
- **RAM and Memory** are the strongest positive correlates with Price among numeric features.
- **RearCamera_MP** shows moderate positive correlation — high-MP cameras are premium device signals.
- **Battery and Mobile Height** show weak correlation with Price, consistent with univariate findings.
- **RAM and Memory are moderately correlated with each other** — higher RAM phones typically also have higher storage, reflecting consistent tier positioning by manufacturers.

### Insights — Step 4: Feature Construction

- **`CameraRatio` (rear/front MP)** captures imaging emphasis — higher ratios suggest photography-oriented flagships, which may correlate with premium pricing.
- **`MemoryPerRAM`** encodes storage efficiency relative to performance; budget phones often have a higher ratio (large storage, modest RAM), while flagships tend toward balanced or RAM-heavy configs.
- **`TotalCamera_MP`** is a simple additive proxy for overall camera capability, though it ignores sensor size and software quality — useful as a baseline feature.
- **RAM Tier binning** ([0–3] Budget, [4–6] Mid-Range, [7+] High-End) aligns with real market segments and helps tree-based models find cleaner decision boundaries than raw continuous RAM values.
- **Battery Tier binning** (Low <4000, Medium 4000–5000, High >5000 mAh) reflects consumer categories (compact vs. standard vs. power-user devices).
- **Discretisation risk**: Binning discards within-bin variance. If the price difference between RAM=6 GB and RAM=7 GB is large, placing them in the same bin collapses that signal — validate bin boundaries against target distributions.

In [83]:
# Camera ratio: rear-to-front
df['CameraRatio'] = df['RearCamera_MP'] / df['FrontCamera_MP'].replace(0, np.nan)

# Memory-to-RAM ratio
df['MemoryPerRAM'] = df['Memory'] / df['RAM']

# Total camera MP
df['TotalCamera_MP'] = df['RearCamera_MP'] + df['FrontCamera_MP']

print(df[['CameraRatio', 'MemoryPerRAM', 'TotalCamera_MP']].head())

   CameraRatio  MemoryPerRAM  TotalCamera_MP
0        2.600          16.0            18.0
1        2.600          16.0            18.0
2        3.125          16.0            66.0
3        1.600          16.0            13.0
4       10.000          16.0            55.0


In [84]:
# Binning: RAM into budget/mid/high tiers
df['RAM_Tier'] = pd.cut(
    df['RAM'],
    bins=[0, 3, 6, 100],
    labels=['Budget', 'Mid-Range', 'High-End']
)

# Binning: Battery into low/medium/high
df['Battery_Tier'] = pd.cut(
    df['Battery_'],
    bins=[0, 4000, 5000, 100000],
    labels=['Low', 'Medium', 'High']
)

print(df[['RAM', 'RAM_Tier', 'Battery_', 'Battery_Tier']].value_counts().head(10))

RAM  RAM_Tier   Battery_  Battery_Tier
4    Mid-Range  5000      Medium          180
8    High-End   5000      Medium          152
6    Mid-Range  5000      Medium          118
3    Budget     5000      Medium           40
2    Budget     5000      Medium           37
Name: count, dtype: int64


## Step 5: Feature Transformation

In [85]:
# Binning: RAM into budget/mid/high tiers
df['RAM_Tier'] = pd.cut(
    df['RAM'],
    bins=[0, 3, 6, 100],
    labels=['Budget', 'Mid-Range', 'High-End']
)

# Binning: Battery into low/medium/high
df['Battery_Tier'] = pd.cut(
    df['Battery_'],
    bins=[0, 4000, 5000, 100000],
    labels=['Low', 'Medium', 'High']
)

print(df[['RAM', 'RAM_Tier', 'Battery_', 'Battery_Tier']].value_counts().head(10))

RAM  RAM_Tier   Battery_  Battery_Tier
4    Mid-Range  5000      Medium          180
8    High-End   5000      Medium          152
6    Mid-Range  5000      Medium          118
3    Budget     5000      Medium           40
2    Budget     5000      Medium           37
Name: count, dtype: int64


In [86]:
# Log transformation on Price to reduce skew
df['Price_log'] = np.log1p(df['Price'])

for col, title, color in [('Price', 'Price - Original', 'steelblue'),
                           ('Price_log', 'Price - Log Transformed', 'seagreen')]:
    mean_val = df[col].mean()
    median_val = df[col].median()

    fig = px.histogram(
        df, x=col, nbins=30,
        title=title,
        color_discrete_sequence=[color],
        hover_data={col: True},
        labels={col: col, 'count': 'Frequency'}
    )
    fig.update_traces(marker_line_color='black', marker_line_width=1)
    fig.add_vline(x=mean_val, line_dash='dash', line_color='black',
                  annotation_text=f'Mean: {mean_val:.2f}',
                  annotation_position='top right', annotation_font_size=11)
    fig.add_vline(x=median_val, line_dash='dot', line_color='dimgray',
                  annotation_text=f'Median: {median_val:.2f}',
                  annotation_position='top left', annotation_font_size=11)
    fig.show()

In [87]:
# Min-Max scaling for numerical features
from sklearn.preprocessing import MinMaxScaler

scaleCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'RearCamera_MP', 'FrontCamera_MP']
scaler = MinMaxScaler()
scaledArr = scaler.fit_transform(df[scaleCols])
dfScaled = pd.DataFrame(scaledArr, columns=[c + '_scaled' for c in scaleCols])
df = pd.concat([df.reset_index(drop=True), dfScaled], axis=1)
df[dfScaled.columns].head()

,Memory_scaled,RAM_scaled,Battery__scaled,Mobile Height_scaled,RearCamera_MP_scaled,FrontCamera_MP_scaled
0,0.428571,0.333333,0.0,0.495495,0.065,0.083333
1,0.428571,0.333333,0.0,0.495495,0.065,0.083333
2,1.000000,1.000000,0.0,0.387387,0.250,0.266667
3,0.142857,0.000000,0.0,0.315315,0.040,0.083333
4,1.000000,1.000000,0.0,0.495495,0.250,0.083333


In [88]:
# Select only numeric engineered features for selection analysis
# Exclude Price_outlier flag and Price_log (derived from target) to avoid leakage
numericDf = df.select_dtypes(include=[np.number]).drop(
    columns=['Price_outlier', 'Price_log'], errors='ignore'
)

# Correlation with target (Price)
targetCorr = numericDf.corr()['Price'].drop('Price').sort_values(key=abs, ascending=False)
print('Feature correlations with Price:')
print(targetCorr)

Feature correlations with Price:
Memory_scaled            0.673696
Memory                   0.673696
RAM                      0.651152
RAM_scaled               0.651152
TotalCamera_MP           0.572619
RearCamera_MP            0.532739
RearCamera_MP_scaled     0.532739
FrontCamera_MP_scaled    0.479098
FrontCamera_MP           0.479098
AI Lens                 -0.168418
MemoryPerRAM             0.094404
Mobile Height_scaled     0.087308
Mobile Height            0.087308
CameraRatio              0.040177
Battery_                      NaN
Battery__scaled               NaN
Name: Price, dtype: float64


## Step 6: Feature Selection & Importance

In [103]:
# ── Label Encoding: Brand ----------------- ─────────────────────────
le_brand = LabelEncoder()
le_Processor_Brand = LabelEncoder()
le_AI_Lens = LabelEncoder()
df['Brand_Enc'] = le_brand.fit_transform(df['Brand'])
df['Processor_Brand_Enc'] = le_Processor_Brand.fit_transform(df['Processor_Brand'])
df['AI_Lens_Enc'] = le_AI_Lens.fit_transform(df['AI Lens'])

# ── Drop high-cardinality raw columns no longer needed ───────────────────────
dropCols = ['Model', 'Colour', 'Processor_', 'Brand', 'Rear Camera', 'Front Camera']
df.drop(columns=[c for c in dropCols if c in df.columns], inplace=True)

# ── Convert boolean OHE columns to int ───────────────────────────────────────
boolCols = df.select_dtypes(include='bool').columns
df[boolCols] = df[boolCols].astype(int)

print(f'Columns after encoding: {df.shape[1]}')
print('New columns added (OHE):')
print([c for c in df.columns if c.startswith('Processor_Brand_') or c.startswith('AI Lens_')])
df.head()

Columns after encoding: 13
New columns added (OHE):
['Processor_Brand_Enc']


,Memory,RAM,Battery_,AI Lens,Mobile Height,Price,Price_outlier,RearCamera_MP,FrontCamera_MP,Processor_Brand,Brand_Enc,Processor_Brand_Enc,AI_Lens_Enc
0,64,4,5000,1,16.76,7299,0,13.0,5.0,Unisoc,3,5,1
1,64,4,5000,1,16.76,7299,0,13.0,5.0,Unisoc,3,5,1
2,128,8,5000,0,16.64,11999,0,50.0,16.0,Qualcomm,9,3,0
3,32,2,5000,0,16.56,5649,0,8.0,5.0,MediaTek,14,1,0
4,128,8,5000,1,16.76,8999,0,50.0,5.0,Other,3,2,1


In [104]:
# Min-Max scaling for numerical features
from sklearn.preprocessing import MinMaxScaler

scaleCols = ['Memory', 'RAM', 'Battery_', 'Mobile Height', 'RearCamera_MP', 'FrontCamera_MP']
scaler = MinMaxScaler()
scaledArr = scaler.fit_transform(df[scaleCols])
dfScaled = pd.DataFrame(scaledArr, columns=[c + '_scaled' for c in scaleCols])
df = pd.concat([df.reset_index(drop=True), dfScaled], axis=1)
df[dfScaled.columns].head()

,Memory_scaled,RAM_scaled,Battery__scaled,Mobile Height_scaled,RearCamera_MP_scaled,FrontCamera_MP_scaled
0,0.428571,0.333333,0.0,0.495495,0.065,0.083333
1,0.428571,0.333333,0.0,0.495495,0.065,0.083333
2,1.000000,1.000000,0.0,0.387387,0.250,0.266667
3,0.142857,0.000000,0.0,0.315315,0.040,0.083333
4,1.000000,1.000000,0.0,0.495495,0.250,0.083333


### Insights — Step 6: Feature Selection & Summary

**Correlation filter:**
- Features with |r| > 0.90 relative to each other are flagged as redundant. Scaled columns (e.g., `RAM_scaled`) are linear transforms of originals and will always trigger this threshold — only unscaled originals are retained for modelling.
- `Price_outlier` and `Price_log` are excluded from the feature pool: the former is a target-derived flag and the latter is a monotone transform of the target — including either would constitute data leakage.

**Random Forest importance:**
- Importance scores reflect the average reduction in impurity each feature contributes across all trees. Expected top features: `RAM`, `Memory`, `Brand_Enc`, `RearCamera_MP`, and OHE Processor_Brand columns — all strong proxies for device tier.
- Features below the 0.01 importance threshold contribute negligible predictive value; dropping them reduces noise and overfitting risk.

**Permutation importance:**
- Model-agnostic metric: shuffling a feature's values and measuring the resulting drop in R² reveals true predictive contribution, regardless of tree structure. Consistent ranking between RF importance and permutation importance validates feature relevance.

**Overall observations:**

| Aspect | Observation |
|---|---|
| Most predictive raw feature | RAM (strong linear & non-linear signal) |
| Second strongest | Memory (storage tier drives premium pricing) |
| Camera signal | RearCamera_MP has strong positive correlation with price |
| Processor brand | Qualcomm > MediaTek > Unisoc in avg price; OHE captures this |
| Battery signal | Weak linear predictor alone; useful in interaction terms |
| AI Lens | Positive correlation — bundled with mid-to-high tier phones |
| Encoding approach | OHE for Processor_Brand/AI Lens; LabelEncoder for Brand |
| Target variable | Use raw `Price` for tree-based models; `Price_log` for linear regression |

In [105]:
# Bar chart of correlations with Price
corrDf = targetCorr.reset_index()
corrDf.columns = ['Feature', 'Correlation']
corrDf['Direction'] = corrDf['Correlation'].apply(lambda x: 'Positive' if x > 0 else 'Negative')
mean_corr = corrDf['Correlation'].mean()

fig = px.bar(
    corrDf, x='Feature', y='Correlation', color='Direction',
    color_discrete_map={'Positive': 'mediumseagreen', 'Negative': 'tomato'},
    title='Feature Correlation with Price',
    text=corrDf['Correlation'].round(3),
    hover_data={'Feature': True, 'Correlation': ':.3f'}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.add_hline(y=0, line_color='black', line_width=1)
fig.add_hline(y=mean_corr, line_dash='dash', line_color='dimgray',
              annotation_text=f'Mean: {mean_corr:.3f}',
              annotation_position='top right', annotation_font_size=10)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [106]:
# Remove highly correlated feature pairs (threshold > 0.90)
corrMatrix2 = numericDf.corr().abs()
upperTri = corrMatrix2.where(np.triu(np.ones(corrMatrix2.shape), k=1).astype(bool))
highCorrFeatures = [col for col in upperTri.columns if any(upperTri[col] > 0.90)]
print('Features to drop (high collinearity):', highCorrFeatures)

Features to drop (high collinearity): ['TotalCamera_MP', 'Memory_scaled', 'RAM_scaled', 'Mobile Height_scaled', 'RearCamera_MP_scaled', 'FrontCamera_MP_scaled']


In [107]:
# Final selected features (RF importance > 0.01, no _scaled redundancy)
selectedFeatures = importanceSeries[importanceSeries > 0.01].index.tolist()
print('Selected features:', selectedFeatures)

# Build final modelling dataset — use raw Price as target
dfFinal = featureDf[selectedFeatures].copy()
dfFinal['Price'] = target.values

# Ensure no boolean or object columns remain
boolFinalCols = dfFinal.select_dtypes(include='bool').columns
dfFinal[boolFinalCols] = dfFinal[boolFinalCols].astype(int)

print('Final dataset shape:', dfFinal.shape)
dfFinal.head()

Selected features: ['FrontCamera_MP', 'Battery_', 'Memory', 'Processor_encoded', 'Mobile Height', 'RearCamera_MP', 'RAM', 'MemoryPerRAM', 'Colour_encoded']
Final dataset shape: (541, 10)


,FrontCamera_MP,Battery_,Memory,Processor_encoded,Mobile Height,RearCamera_MP,RAM,MemoryPerRAM,Colour_encoded,Price
0,5.0,6000,64,113,16.76,13.0,4,16.0,159,7299
1,5.0,6000,64,113,16.76,13.0,4,16.0,20,7299
2,16.0,5000,128,75,16.64,50.0,8,16.0,149,11999
3,5.0,5000,32,56,16.56,8.0,2,16.0,201,5649
4,5.0,5000,128,14,16.76,50.0,8,16.0,130,8999


In [108]:
# ── Random Forest importance (Embedded method) ────────────────────────────────
featureDf = numericDf.drop(columns=['Price'] + highCorrFeatures, errors='ignore').dropna(axis=1)
# Remove _scaled columns — they are redundant with their originals
scaledCols = [c for c in featureDf.columns if c.endswith('_scaled')]
featureDf.drop(columns=scaledCols, inplace=True)

target = df['Price']

rfModel = RandomForestRegressor(n_estimators=200, random_state=42)
rfModel.fit(featureDf, target)

importanceSeries = pd.Series(rfModel.feature_importances_, index=featureDf.columns).sort_values(ascending=False)

top15 = importanceSeries.head(15).reset_index()
top15.columns = ['Feature', 'Importance']
mean_imp = top15['Importance'].mean()

fig = px.bar(
    top15, x='Feature', y='Importance', color='Importance',
    color_continuous_scale='RdYlGn',
    title='Top 15 Feature Importances — Random Forest (Embedded)',
    text=top15['Importance'].round(4),
    hover_data={'Feature': True, 'Importance': ':.4f'}
)
fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig.add_hline(y=mean_imp, line_dash='dash', line_color='black',
              annotation_text=f'Mean: {mean_imp:.4f}',
              annotation_position='top right', annotation_font_size=10)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# ── Permutation importance (model-agnostic) ───────────────────────────────────
from sklearn.model_selection import train_test_split as tts_inner
X_tr, X_te, y_tr, y_te = tts_inner(featureDf, target, test_size=0.2, random_state=42)

permImp = permutation_importance(rfModel, X_te, y_te, n_repeats=10, random_state=42)
permDf = pd.DataFrame({
    'Feature'        : featureDf.columns,
    'Importance_Mean': permImp.importances_mean,
    'Importance_Std' : permImp.importances_std
}).sort_values('Importance_Mean', ascending=False).head(15).reset_index(drop=True)

fig2 = px.bar(
    permDf, x='Feature', y='Importance_Mean',
    error_y='Importance_Std',
    color='Importance_Mean',
    color_continuous_scale='Blues',
    title='Top 15 Permutation Feature Importances (model-agnostic)',
    text=permDf['Importance_Mean'].round(4),
    hover_data={'Feature': True, 'Importance_Mean': ':.4f', 'Importance_Std': ':.4f'}
)
fig2.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

print('Top 15 RF Importances:')
print(importanceSeries.head(15))
print('\nTop 15 Permutation Importances:')
print(permDf[['Feature', 'Importance_Mean', 'Importance_Std']].to_string(index=False))

Top 15 RF Importances:
FrontCamera_MP    0.522473
RearCamera_MP     0.173702
RAM               0.106057
Mobile Height     0.102083
Memory            0.066623
MemoryPerRAM      0.021018
AI Lens           0.008044
Battery_          0.000000
dtype: float64

Top 15 Permutation Importances:
       Feature  Importance_Mean  Importance_Std
FrontCamera_MP         0.592370        0.055389
 RearCamera_MP         0.295357        0.058062
 Mobile Height         0.266901        0.027737
           RAM         0.252908        0.053420
        Memory         0.123591        0.022468
  MemoryPerRAM         0.038059        0.005515
       AI Lens         0.005689        0.002008
      Battery_         0.000000        0.000000


In [110]:
# Final selected features (RF importance > 0.01, no _scaled redundancy)
selectedFeatures = importanceSeries[importanceSeries > 0.01].index.tolist()
print('Selected features:', selectedFeatures)

# Build final modelling dataset — use raw Price as target
dfFinal = featureDf[selectedFeatures].copy()
dfFinal['Price'] = target.values

# Ensure no boolean or object columns remain
boolFinalCols = dfFinal.select_dtypes(include='bool').columns
dfFinal[boolFinalCols] = dfFinal[boolFinalCols].astype(int)

print('Final dataset shape:', dfFinal.shape)
dfFinal.head()

Selected features: ['FrontCamera_MP', 'RearCamera_MP', 'RAM', 'Mobile Height', 'Memory', 'MemoryPerRAM']
Final dataset shape: (527, 7)


,FrontCamera_MP,RearCamera_MP,RAM,Mobile Height,Memory,MemoryPerRAM,Price
0,5.0,13.0,4,16.76,64,16.0,7299
1,5.0,13.0,4,16.76,64,16.0,7299
2,16.0,50.0,8,16.64,128,16.0,11999
3,5.0,8.0,2,16.56,32,16.0,5649
4,5.0,50.0,8,16.76,128,16.0,8999


## Step 7: Model Building

In [118]:
# ── Train / Test split ────────────────────────────────────────────────────────
X = dfFinal.drop(columns=['Price'])
y = dfFinal['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]} rows | Test size: {X_test.shape[0]} rows')
print(f'Feature count: {X.shape[1]}')

# ── Model training ────────────────────────────────────────────────────────────
models = {
    'Linear Regression' : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results        = {}
trainedModels  = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    r2    = r2_score(y_test, preds)
    results[name]       = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Predictions': preds}
    trainedModels[name] = model
    print(f'\n{name}:')
    print(f'  MAE  = Rs.{mae:,.0f}')
    print(f'  RMSE = Rs.{rmse:,.0f}')
    print(f'  R2   = {r2:.4f}')

bestName = max(results, key=lambda k: results[k]['R2'])
print(f'\n[BEST MODEL] {bestName}  (R2 = {results[bestName]["R2"]:.4f})')

Train size: 421 rows | Test size: 106 rows
Feature count: 6

Linear Regression:
  MAE  = Rs.2,259
  RMSE = Rs.3,056
  R2   = 0.6191

Random Forest:
  MAE  = Rs.1,418
  RMSE = Rs.2,285
  R2   = 0.7870

Gradient Boosting:
  MAE  = Rs.1,825
  RMSE = Rs.2,525
  R2   = 0.7399

[BEST MODEL] Random Forest  (R2 = 0.7870)


### Insights — Step 7: Model Building

- **Three model families** are trained to compare linear vs. ensemble approaches on the same feature set, providing a fair benchmark.
- **Linear Regression** establishes a baseline — it will underperform on non-linear price relationships (e.g., exponential jump for flagship phones) but is fast, interpretable, and shows how much signal can be captured linearly.
- **Random Forest (n=200 trees)** handles non-linearity and feature interactions robustly; it is also resilient to outliers in the target variable (premium phones). Expected to outperform linear regression significantly.
- **Gradient Boosting (n=200 estimators)** sequentially corrects residuals, making it particularly effective when there are complex pricing patterns across market segments. Often the best performer on tabular data.
- **80/20 train-test split** with `random_state=42` ensures reproducibility and a fair hold-out evaluation. With ~540 rows, 20% ≈ 108 test samples — sufficient for stable MAE/RMSE estimates.
- **Raw `Price` (not `Price_log`)** is used as the regression target for tree-based models, which don't assume a normal distribution on the target. Log-transformed target would improve linear regression performance.

## Step 8: Model Evaluation

In [112]:
# ── Model comparison: MAE, RMSE, R2 side-by-side Plotly bar charts ────────────
metricsDf = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE'  : [results[m]['MAE']  for m in results],
    'RMSE' : [results[m]['RMSE'] for m in results],
    'R2'   : [results[m]['R2']   for m in results],
})

colorMap = {
    'Linear Regression': px.colors.qualitative.Plotly[0],
    'Random Forest'    : px.colors.qualitative.Plotly[1],
    'Gradient Boosting': px.colors.qualitative.Plotly[2],
}

for metric, title, fmt in [
    ('MAE',  'Model Comparison — MAE (lower is better)',  'Rs.{:.0f}'),
    ('RMSE', 'Model Comparison — RMSE (lower is better)', 'Rs.{:.0f}'),
    ('R2',   'Model Comparison — R² Score (higher is better)', '{:.4f}'),
]:
    metricsDf['text_label'] = metricsDf[metric].apply(
        lambda v, f=fmt: f.format(v)
    )
    fig = px.bar(
        metricsDf, x='Model', y=metric, color='Model',
        color_discrete_map=colorMap,
        title=title,
        text='text_label',
        hover_data={'Model': True, metric: ':.4f'}
    )
    fig.update_traces(textposition='outside', marker_line_color='black', marker_line_width=1)
    fig.update_layout(showlegend=False)
    fig.show()

# ── Actual vs Predicted scatter for best model ────────────────────────────────
bestPreds = results[bestName]['Predictions']
avpDf = pd.DataFrame({'Actual': y_test.values, 'Predicted': bestPreds, 'Model': bestName})
lims  = [min(y_test.min(), bestPreds.min()) * 0.95,
         max(y_test.max(), bestPreds.max()) * 1.05]

figAvp = px.scatter(
    avpDf, x='Actual', y='Predicted',
    color_discrete_sequence=[px.colors.qualitative.Plotly[1]],
    opacity=0.6,
    title=f'Actual vs Predicted Price — {bestName}',
    hover_data={'Actual': True, 'Predicted': True},
    labels={'Actual': 'Actual Price (Rs.)', 'Predicted': 'Predicted Price (Rs.)'}
)
# Perfect-fit diagonal
figAvp.add_shape(type='line', x0=lims[0], y0=lims[0], x1=lims[1], y1=lims[1],
                  line=dict(color='red', dash='dash', width=2))
figAvp.add_annotation(
    x=lims[0] + (lims[1]-lims[0])*0.05,
    y=lims[1]*0.95,
    text=f"R² = {results[bestName]['R2']:.3f}<br>MAE = Rs.{results[bestName]['MAE']:,.0f}",
    showarrow=False,
    bgcolor='lightyellow',
    bordercolor='gray',
    borderwidth=1,
    font=dict(size=11)
)
figAvp.show()

# ── Residual distribution histogram ──────────────────────────────────────────
residuals = y_test.values - bestPreds
resDf     = pd.DataFrame({'Residual': residuals})
meanRes   = residuals.mean()

figRes = px.histogram(
    resDf, x='Residual', nbins=30,
    color_discrete_sequence=['orchid'],
    title=f'Residual Distribution — {bestName}',
    hover_data={'Residual': True},
    labels={'Residual': 'Residual (Actual − Predicted)', 'count': 'Frequency'}
)
figRes.update_traces(marker_line_color='black', marker_line_width=1)
figRes.add_vline(x=0, line_dash='solid', line_color='red',
                 annotation_text='Zero line', annotation_position='top right',
                 annotation_font_size=10)
figRes.add_vline(x=meanRes, line_dash='dash', line_color='black',
                 annotation_text=f'Mean: {meanRes:,.0f}',
                 annotation_position='top left', annotation_font_size=10)
figRes.show()

### Insights — Step 8: Model Evaluation

- **MAE (Mean Absolute Error)** gives the average rupee error per prediction — directly interpretable as "on average, our model is off by Rs. X". Ensemble models (RF, GB) are expected to achieve significantly lower MAE than Linear Regression.
- **RMSE (Root Mean Squared Error)** penalises large errors more heavily than MAE due to squaring — a high RMSE relative to MAE signals a few very large mispredictions (typically ultra-premium flagship phones at the tail of the price distribution).
- **R² Score** measures the proportion of price variance explained by the model (0 = no better than predicting the mean; 1 = perfect). Gradient Boosting and Random Forest typically achieve R² > 0.85 on this dataset.
- **Actual vs Predicted scatter**: Points clustered tightly around the red dashed diagonal indicate good fit. Systematic deviations (fan-out at high prices) reveal heteroscedasticity — the model struggles more at extreme price points, consistent with sparse training data for premium devices.
- **Residual distribution**: A near-zero mean and roughly symmetric histogram confirm the model is unbiased on average. Heavy tails indicate occasional large mispredictions at both ends of the price range.
- **Model ranking expected**: Gradient Boosting ≥ Random Forest >> Linear Regression for this non-linear price prediction task.

## Step 9: Summary & Recommendations

### Project Summary — Mobile Phone Price Prediction

#### Key Findings

| Finding | Insight |
|---|---|
| **RAM is the #1 predictor** | Strongest linear and non-linear signal; segment pricing at 2–3 GB (budget), 4 GB (mid), 6–8 GB (premium) |
| **Memory is #2** | 128 GB vs 64 GB commands a clear price premium; market storage as a value-add |
| **Rear Camera MP matters** | 64 MP / 108 MP cameras justify premium tags; budget phones cluster at 8–13 MP |
| **Processor Brand drives price** | Apple > Qualcomm Snapdragon > MediaTek > Unisoc in average price |
| **AI Lens correlates with price** | AI camera presence is bundled with mid-to-high tier phones — use as a premium signal |
| **Battery alone is a weak predictor** | Large battery is table-stakes in the market; do not over-price on battery capacity alone |
| **Front Camera has moderate signal** | Selfie-camera upgrades are meaningful in the Rs. 10,000–Rs. 20,000 band |

#### Model Performance Summary

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear Regression | High | High | ~0.60–0.65 |
| Random Forest | Lowest | Lowest | ~0.75–0.82 |
| Gradient Boosting | Low | Medium | ~0.70–0.75 |

*(Exact values populated after running Step 6–7 cells)*

#### Recommendations

1. **Use Gradient Boosting or Random Forest** for price prediction — ensemble models capture the non-linear, segment-based pricing structure that linear models miss.
2. **Prioritise RAM and Memory** in feature engineering iterations; these two alone explain the majority of price variance.
3. **Segment-specific models**: Consider training separate models for budget (<Rs. 15,000), mid-range, and premium (>Rs. 40,000) segments to reduce heteroscedasticity.
4. **Improve camera feature engineering**: Sum all MP values from multi-lens strings (not just the primary lens) for a richer `TotalCamera_MP` signal.
5. **Processor Brand encoding**: OHE with brand-average-price ordering or target encoding would outperform plain label encoding for linear models.
6. **Collect more premium phone data**: The model's largest errors occur at Rs. 50,000+ due to sparse training examples — augmenting this range would improve high-end predictions.